# PipeGuard-IC
## An Open-Source Self-Calibrating Programmable Analog Front End for Edge Pipeline-Leak Monitoring

**ISSCC 2027 IEEE SSCS Code-a-Chip — development notebook**

**Lead:** Ayoola Timilehin Israel, Mechatronics Engineering, Federal University of Technology, Minna, Nigeria

> This notebook separates design targets, analytical exploration, and circuit-simulation evidence. The current milestone is a reproducible signal-model and specification baseline. Transistor-level SKY130/ngspice results will be added only after validation.

## 1. Motivation and proposed signal chain

Weak vibration or acoustic signatures from a pipeline-mounted sensor can be obscured by pump tones, structural vibration, electrical interference, and sensor noise. PipeGuard-IC investigates a low-power analog front end that conditions this signal before ADC sampling and edge inference.

`Sensor → Low-noise input stage → Programmable gain → Tunable band-pass filter → ADC-ready output → Edge classifier`

The notebook-driven flow will eventually automate circuit generation, ngspice analysis, parameter sweeps, corner checks, and presentation of the main performance metrics.

## 2. Reproducibility

Install the pinned Python dependencies from `requirements.txt`. The future SPICE milestone will add explicit ngspice, Xschem, SKY130 PDK, and operating-system setup checks.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal

SEED = 27
rng = np.random.default_rng(SEED)
print(f'Python: {sys.version.split()[0]}')
print(f'NumPy: {np.__version__}')
print('PipeGuard-IC analytical environment ready.')

## 3. Initial target specifications

These are engineering targets for the first circuit iteration, not measured or simulated silicon results. They will be revised after literature review and transistor-level feasibility analysis.

In [ ]:
targets = pd.DataFrame({
    'Metric': ['Supply voltage', 'Programmable gain', 'Passband', 'Input-referred noise', 'Power', 'Output interface'],
    'Initial target': ['1.8 V', '20/40/60 dB', '0.3–8 kHz (tunable)', 'To be established', '< 1 mW', 'ADC-ready, centred at VCM'],
    'Verification method': ['DC operating point', 'AC sweep', 'AC sweep', 'ngspice noise analysis', 'Supply-current integration', 'Transient analysis']
})
targets

## 4. Reproducible sensor and interference model

The synthetic example creates a known leak-like broadband component plus low-frequency machinery vibration, a narrow interference tone, and Gaussian sensor noise. It is a development stimulus—not field evidence—and makes the analytical workflow reproducible without a private dataset.

In [ ]:
fs = 50_000
duration = 0.20
t = np.arange(0, duration, 1/fs)

# Leak-like component: band-limited stochastic vibration
sos_leak = signal.butter(4, [900, 4200], btype='bandpass', fs=fs, output='sos')
leak_clean = signal.sosfiltfilt(sos_leak, rng.normal(size=t.size))
leak_clean = 2.5e-3 * leak_clean / np.std(leak_clean)

machinery = 8e-3*np.sin(2*np.pi*120*t) + 4e-3*np.sin(2*np.pi*240*t)
interference = 3e-3*np.sin(2*np.pi*10_000*t)
sensor_noise = rng.normal(scale=2.5e-3, size=t.size)
raw_signal = leak_clean + machinery + interference + sensor_noise

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
window = t < 0.025
ax[0].plot(t[window]*1e3, raw_signal[window]*1e3, lw=1, color='#0B4F6C')
ax[0].set(title='Synthetic raw sensor signal', xlabel='Time (ms)', ylabel='Amplitude (mV)')
f, pxx = signal.welch(raw_signal, fs=fs, nperseg=2048)
ax[1].semilogx(f[1:], 10*np.log10(pxx[1:]+1e-30), color='#01BAEF')
ax[1].set(title='Raw-signal power spectral density', xlabel='Frequency (Hz)', ylabel='PSD (dB/Hz)', xlim=(20, fs/2))
for a in ax: a.grid(alpha=.25)
plt.tight_layout()

## 5. Analytical front-end baseline

Before transistor-level design, an ideal fourth-order band-pass response provides a reference for the desired conditioning behaviour. This is not presented as circuit performance.

In [ ]:
f_low, f_high = 300, 8_000
sos_afe = signal.butter(4, [f_low, f_high], btype='bandpass', fs=fs, output='sos')
conditioned = signal.sosfiltfilt(sos_afe, raw_signal)

def snr_db(reference, estimate):
    error = estimate - reference
    return 10*np.log10(np.mean(reference**2) / np.mean(error**2))

baseline_snr = snr_db(leak_clean, raw_signal)
conditioned_reference = signal.sosfiltfilt(sos_afe, leak_clean)
conditioned_snr = snr_db(conditioned_reference, conditioned)
metrics = pd.DataFrame({
    'Analytical metric': ['Input SNR', 'Conditioned SNR', 'Illustrative SNR change'],
    'Value (dB)': [baseline_snr, conditioned_snr, conditioned_snr-baseline_snr]
})
metrics.round(2)

In [ ]:
w, h = signal.sosfreqz(sos_afe, worN=4096, fs=fs)
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].semilogx(w[1:], 20*np.log10(np.maximum(np.abs(h[1:]), 1e-12)), color='#FF7F11', lw=2)
ax[0].axvspan(f_low, f_high, color='#01BAEF', alpha=.12, label='Target passband')
ax[0].set(title='Ideal analytical band-pass response', xlabel='Frequency (Hz)', ylabel='Magnitude (dB)', ylim=(-80, 5))
ax[0].legend()
ax[1].plot(t[window]*1e3, raw_signal[window]*1e3, label='Raw', alpha=.55, color='#7A7A7A')
ax[1].plot(t[window]*1e3, conditioned[window]*1e3, label='Conditioned', lw=1.2, color='#0B4F6C')
ax[1].set(title='Before and after analytical conditioning', xlabel='Time (ms)', ylabel='Amplitude (mV)')
ax[1].legend()
for a in ax: a.grid(alpha=.25)
plt.tight_layout()

## 6. Planned circuit-verification milestones

1. Select and justify the sensor-equivalent impedance and signal range.
2. Design the SKY130 low-noise input stage and bias network.
3. Implement programmable gain without violating headroom or stability constraints.
4. Implement and tune the integrated filter architecture.
5. Run DC, AC, transient, noise, distortion, and power analyses in ngspice.
6. Automate component/device sweeps and multi-objective trade-off plots.
7. Run process, voltage, temperature, and mismatch checks.
8. Attempt reusable layout generation for the best-performing block.
9. Add limitations, reproducibility checks, references, and a final datasheet summary.

### Current limitation

This milestone contains analytical signal-processing evidence only. It must not be interpreted as transistor-level performance. The next milestone will introduce open-PDK circuit netlists and independently reproducible ngspice outputs.

## 7. First open-PDK circuit milestone

The repository now includes a parameterized passive piezoelectric sensor model and an exploratory 1.8 V SKY130 differential amplifier. The amplifier uses an NMOS input pair, NMOS tail device, and PMOS current-mirror load. This topology is a starting point for evidence-driven iteration—not a performance claim.

Two testbenches and a guarded Python runner are included:

- `circuits/testbench_dc.spice` checks the operating point and supply current.
- `circuits/testbench_ac.spice` applies a 1 V differential small signal and records gain and phase.
- `scripts/run_ngspice.py` finds the SKY130 model library, runs ngspice, and refuses to report PASS without output data.
- `scripts/analyze_spice.py` validates the generated files and creates a summary table and Bode plot.

Run these commands from the project directory after installing ngspice and the SKY130A open PDK:

```bash
python scripts/run_ngspice.py --analysis all
python scripts/analyze_spice.py
```


In [ ]:
from pathlib import Path
project_root = Path.cwd()
expected_assets = [
    'models/piezo_sensor_model.spice',
    'circuits/sky130_lna.spice',
    'circuits/testbench_dc.spice',
    'circuits/testbench_ac.spice',
    'scripts/run_ngspice.py',
    'scripts/analyze_spice.py',
]
pd.DataFrame({'asset': expected_assets, 'present': [(project_root / p).is_file() for p in expected_assets]})


### Evidence gate

No transistor-level metric is shown in this notebook until `results/dc_operating_point.dat` and `results/ac_response.dat` are generated by ngspice using the SKY130 model library. This prevents analytical targets from being mistaken for simulated or measured results. The immediate next iteration is to establish valid device operating regions, then add noise and transient verification before programmable gain and filtering.


## 8. Bias, noise, and transient verification plan

The second circuit milestone expands verification before adding architectural complexity. A tail-bias sweep searches for output headroom and current feasibility; noise analysis estimates input-referred noise over the 300 Hz–8 kHz target band; and transient analysis applies a 2 kHz, 1 mV-peak differential stimulus to estimate time-domain gain and average power.

These tests are available through the same evidence-gated runner:

```bash
python scripts/run_ngspice.py --analysis bias
python scripts/run_ngspice.py --analysis noise
python scripts/run_ngspice.py --analysis transient
python scripts/analyze_spice.py
```

The analysis script creates `summary.csv` and any available AC, bias, noise, and transient plots. Numerical claims should be copied into this notebook only after the corresponding raw result and ngspice log are committed.


In [ ]:
results_dir = Path('results')
result_manifest = pd.DataFrame({
    'analysis': ['DC operating point', 'AC response', 'Bias sweep', 'Noise', 'Transient'],
    'expected file': ['dc_operating_point.dat', 'ac_response.dat', 'bias_sweep.dat', 'noise_response.dat', 'transient_response.dat'],
})
result_manifest['verified output present'] = result_manifest['expected file'].map(lambda name: (results_dir / name).is_file())
result_manifest
